# Assignment 04 · Notebook 02
# Mạng nơ-ron tích chập một chiều hiện thực bằng PyTorch cho bài toán hồi quy giá bất động sản


| Mục | Nội dung |
|---|---|
| Học phần | Phát triển các Hệ thống Thông minh |
| Cơ sở đào tạo | Học viện Công nghệ Bưu chính Viễn thông |
| Sinh viên | **Nguyễn Duy Nghĩa** |
| Mã sinh viên | **B23DCCN600** |
| Lớp | **D23CTPM01** |
| Giảng viên hướng dẫn | **PGS.TS Trần Đình Quế** |
| Học kỳ | Học kỳ 1 năm học 2026 - 2027 |
| Assignment | 04 - Convolutional Neural Networks |
| Miền dữ liệu | `house_price` (hồi quy giá bất động sản Hoa Kỳ) |


---

## Mục tiêu của notebook

Notebook 01 đã xây dựng toàn bộ mạng CNN một chiều từ định nghĩa toán học bằng NumPy thuần, bao gồm
cả phần lan truyền ngược viết tay và đã được xác nhận bằng sai phân hữu hạn. Notebook 02 hiện thực
**đúng kiến trúc đó** bằng PyTorch, với ba mục tiêu:

1. **Tái lập chính xác kiến trúc của hợp đồng** bằng các tầng dựng sẵn `nn.Conv1d`, `nn.ReLU`,
   `nn.MaxPool1d`, `nn.Flatten`, `nn.Linear`, với hàm mất mát `nn.MSELoss` và đầu ra tuyến tính.
   Số tham số phải trùng khớp tuyệt đối với con số 1 377 mà notebook 01 đếm được, vì hai hiện thực
   mô tả cùng một hàm toán học.
2. **Đối chiếu ngữ nghĩa giữa hiện thực thủ công và thư viện**: xác nhận rằng `nn.Conv1d` cũng thực
   hiện tương quan chéo chứ không phải tích chập theo nghĩa giải tích, và rằng `padding=1` với
   $K=3$ tương đương chế độ `same` đã cài trong notebook 01.
3. **So sánh chi phí và lợi ích của tự động vi phân**: PyTorch dựng đồ thị tính toán động và gọi
   `loss.backward()` thay cho toàn bộ phần đạo hàm viết tay, đổi lấy việc mất quyền kiểm soát chi
   tiết từng bước.

Quy trình tiền xử lý, phân chia dữ liệu và hạt giống ngẫu nhiên giữ nguyên tuyệt đối như notebook
01, nhờ đó phép so sánh ba framework ở notebook 03 là một phép so sánh công bằng.

---

## 1. Nhập thư viện và cố định hạt giống ngẫu nhiên

Ngoài các thư viện chung, notebook này nạp thêm PyTorch. Theo hợp đồng, môi trường chỉ có phiên bản
CPU, không có CUDA hay MPS, nên mọi phép tính chạy trên CPU. Với một mô hình chỉ 1 377 tham số và
chuỗi dài 8, đây không phải hạn chế đáng kể: chi phí chuyển dữ liệu qua GPU còn lớn hơn chính phép
tính.

In [ ]:
# ====== Thư viện chuẩn của báo cáo ======
import os, json, time, math, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")                      # backend không cần màn hình, phù hợp nbconvert
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# ====== Hạt giống ngẫu nhiên: cố định để mọi lần chạy tái lập được ======
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ====== Quy ước vẽ hình theo hợp đồng tích hợp (Mục 5.2) ======
plt.rcParams["font.sans-serif"] = ["Segoe UI", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["savefig.facecolor"] = "white"
plt.rcParams["figure.dpi"] = 110
sns.set_style("whitegrid")

# ====== Đường dẫn tương đối tính từ thư mục notebooks/ ======
DATA_PATH = "../data/usa_real_estate_150k.csv"
FIG_DIR   = "../reports/figures"
REP_DIR   = "../reports"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(REP_DIR, exist_ok=True)

print("NumPy      :", np.__version__)
print("pandas     :", pd.__version__)
print("matplotlib :", matplotlib.__version__)
print("seaborn    :", sns.__version__)
print("RANDOM_SEED:", RANDOM_SEED)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Cố định hạt giống cho mọi nguồn ngẫu nhiên của PyTorch
torch.manual_seed(RANDOM_SEED)
torch.use_deterministic_algorithms(False)
# Giới hạn số luồng: mô hình chỉ 1 377 tham số và lô chỉ (256, 16, 8), nên chi phí đồng bộ
# giữa các luồng lớn hơn chính phép tính. Đo thực tế trên máy 32 nhân: 32 luồng mất 40,09 giây
# cho một epoch, còn 4 luồng chỉ mất 2,42 giây, tức nhanh hơn khoảng 16,6 lần.
TORCH_THREADS = min(4, os.cpu_count() or 4)
torch.set_num_threads(TORCH_THREADS)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch        :", torch.__version__)
print("CUDA khả dụng  :", torch.cuda.is_available())
print("Thiết bị dùng  :", DEVICE)
print("Số luồng CPU   :", torch.get_num_threads(), "(giới hạn có chủ đích, xem chú thích trong ô lệnh)")

---

## 2. Nạp dữ liệu và tiền xử lý

Toàn bộ khối lệnh dưới đây **giống hệt notebook 01 từng dòng**. Sự đồng nhất này không phải để tiết
kiệm công sức mà là một yêu cầu phương pháp luận: nếu hai notebook tiền xử lý khác nhau dù chỉ ở
cách điền khuyết `acre_lot`, thì chênh lệch chỉ số giữa NumPy và PyTorch sẽ phản ánh chênh lệch dữ
liệu chứ không phản ánh chênh lệch hiện thực.

Nhắc lại ba quyết định tiền xử lý quan trọng đã được biện luận đầy đủ ở notebook 01:

- **Lấy logarit biến mục tiêu** vì sai số giá bất động sản có tính nhân, không phải tính cộng.
- **Điền khuyết `acre_lot` bằng trung vị** thay vì loại bỏ, vì trường này khuyết 21,6 phần trăm.
- **Chuẩn hóa khớp riêng trên tập huấn luyện** để tránh rò rỉ dữ liệu.

In [ ]:
# ---------- 1. Nạp dữ liệu thô ----------
raw = pd.read_csv(DATA_PATH)
n_raw = len(raw)

# ---------- 2. Loại bản ghi thiếu bốn trường cốt lõi ----------
df = raw.dropna(subset=["price", "house_size", "bed", "bath"]).copy()

# ---------- 3. Lọc khoảng giá trị hợp lệ theo hợp đồng ----------
df = df[(df["price"] >= 10_000) & (df["price"] <= 5_000_000)]
df = df[(df["house_size"] >= 200) & (df["house_size"] <= 20_000)]

# ---------- 4. Xử lý acre_lot thiếu: điền trung vị rồi chặn dưới để lấy log an toàn ----------
ACRE_MEDIAN = df["acre_lot"].median()
df["acre_lot"] = df["acre_lot"].fillna(ACRE_MEDIAN).clip(lower=1e-3)

# ---------- 5. Kỹ thuật đặc trưng: đúng 8 đặc trưng, đúng thứ tự hợp đồng ----------
df["log_house_size"] = np.log(df["house_size"])
df["total_rooms"]    = df["bed"] + df["bath"]
df["bed_bath_prod"]  = df["bed"] * df["bath"]
df["sqft_per_room"]  = df["house_size"] / df["total_rooms"].replace(0, np.nan)
df["bath_bed_ratio"] = df["bath"] / df["bed"].replace(0, np.nan)
df["log_acre_lot"]   = np.log(df["acre_lot"])

# ---------- 6. Mục tiêu hồi quy trên thang logarit ----------
df["log_price"] = np.log(df["price"])

FEATURES = ["log_house_size", "bed", "bath", "total_rooms",
            "bed_bath_prod", "sqft_per_room", "bath_bed_ratio", "log_acre_lot"]
TARGET   = "log_price"

df = df.dropna(subset=FEATURES + [TARGET])
n_clean = len(df)

print(f"Số bản ghi thô      n_raw   = {n_raw:,}")
print(f"Số bản ghi sạch     n_clean = {n_clean:,}")
print(f"Tỷ lệ giữ lại               = {n_clean / n_raw:.4f}")
print(f"Trung vị acre_lot dùng để điền khuyết = {ACRE_MEDIAN}")
print()
print("Thống kê mô tả 8 đặc trưng:")
display(df[FEATURES].describe().T[["mean", "std", "min", "25%", "50%", "75%", "max"]].round(4))

In [ ]:
X_all = df[FEATURES].to_numpy(dtype=np.float64)
y_all = df[TARGET].to_numpy(dtype=np.float64)

# Tách test trước (20%), rồi tách validation từ phần còn lại (20% của 80% = 16% tổng thể)
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X_all, y_all, test_size=0.20, random_state=RANDOM_SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.20, random_state=RANDOM_SEED)

# Chuẩn hóa: fit CHỈ trên train, sau đó transform cho val và test
scaler = StandardScaler().fit(X_train)

# Winsorize sau chuẩn hóa: chặn mọi tọa độ trong khoảng +/- CLIP_SIGMA độ lệch chuẩn.
# Lý do: một số bản ghi có bed_bath_prod tới 1833 (hơn 160 độ lệch chuẩn). Qua hai tầng
# tích chập tuyến tính cộng ReLU, giá trị đó tạo ra dự đoán log_price rất lớn, và sau khi
# lấy np.exp thì sai số USD của vài chục bản ghi lấn át toàn bộ 30 000 mẫu kiểm tra.
CLIP_SIGMA = 5.0
X_train_s = np.clip(scaler.transform(X_train), -CLIP_SIGMA, CLIP_SIGMA)
X_val_s   = np.clip(scaler.transform(X_val),   -CLIP_SIGMA, CLIP_SIGMA)
X_test_s  = np.clip(scaler.transform(X_test),  -CLIP_SIGMA, CLIP_SIGMA)
n_clip_train = int((np.abs(scaler.transform(X_train)) > CLIP_SIGMA).any(axis=1).sum())

# Định dạng chuỗi cho CNN 1 chiều: (N, C_in = 1, L = 8)
Xtr = X_train_s.reshape(-1, 1, 8).astype(np.float32)
Xva = X_val_s.reshape(-1, 1, 8).astype(np.float32)
Xte = X_test_s.reshape(-1, 1, 8).astype(np.float32)
ytr = y_train.reshape(-1, 1).astype(np.float32)
yva = y_val.reshape(-1, 1).astype(np.float32)
yte = y_test.reshape(-1, 1).astype(np.float32)

n_train, n_val, n_test = len(Xtr), len(Xva), len(Xte)
print(f"Train      : {n_train:,} mẫu  ->  tensor {Xtr.shape}")
print(f"Validation : {n_val:,} mẫu  ->  tensor {Xva.shape}")
print(f"Test       : {n_test:,} mẫu  ->  tensor {Xte.shape}")
print()
print("Trung bình sau chuẩn hóa trên train (kỳ vọng xấp xỉ 0):")
print(np.round(Xtr.reshape(-1, 8).mean(axis=0), 6))
print("Độ lệch chuẩn sau chuẩn hóa trên train (kỳ vọng xấp xỉ 1):")
print(np.round(Xtr.reshape(-1, 8).std(axis=0), 6))
print()
print(f"Số mẫu train bị winsorize ở ít nhất một tọa độ: {n_clip_train:,}"
      f"  ({n_clip_train / n_train * 100:.3f}% tập huấn luyện)")
print()
print(f"log_price train: mean = {ytr.mean():.4f}, std = {ytr.std():.4f}")
print(f"Giá tương ứng exp(mean) = {np.exp(ytr.mean()):,.0f} USD")

In [ ]:
def danh_gia_hoi_quy(y_true_log, y_pred_log):
    """Tính đủ bộ chỉ số hồi quy theo hợp đồng Mục 5.3.

    Tham số đầu vào nằm trên THANG LOG. Sai số USD được quy đổi ngược bằng exp().
    """
    yt = np.asarray(y_true_log, dtype=np.float64).ravel()
    yp = np.asarray(y_pred_log, dtype=np.float64).ravel()

    # --- Thang log ---
    err_log  = yp - yt
    rmse_log = float(np.sqrt(np.mean(err_log ** 2)))
    mae_log  = float(np.mean(np.abs(err_log)))
    ss_res   = float(np.sum(err_log ** 2))
    ss_tot   = float(np.sum((yt - yt.mean()) ** 2))
    r2       = float(1.0 - ss_res / ss_tot)

    # --- Quy đổi ngược về USD ---
    yt_usd = np.exp(yt)
    yp_usd = np.exp(yp)
    err_usd  = yp_usd - yt_usd
    rmse_usd = float(np.sqrt(np.mean(err_usd ** 2)))
    mae_usd  = float(np.mean(np.abs(err_usd)))

    return {"rmse_usd": rmse_usd, "mae_usd": mae_usd, "r2": r2,
            "rmse_log": rmse_log, "mae_log": mae_log}


def lay_scatter_sample(y_true_log, y_pred_log, n=200, seed=RANDOM_SEED):
    """Rút 200 điểm ngẫu nhiên (thang log) để hợp đồng dựng biểu đồ tán xạ."""
    rng = np.random.default_rng(seed)
    yt = np.asarray(y_true_log, dtype=np.float64).ravel()
    yp = np.asarray(y_pred_log, dtype=np.float64).ravel()
    idx = rng.choice(len(yt), size=min(n, len(yt)), replace=False)
    return {"y_true": yt[idx].tolist(), "y_pred": yp[idx].tolist()}


print("Đã định nghĩa hai hàm dùng chung: danh_gia_hoi_quy() và lay_scatter_sample().")
print("Bộ chỉ số: rmse_usd, mae_usd, r2, rmse_log, mae_log (theo CONTRACT Mục 5.3).")

### Diễn giải kết quả tiền xử lý

Các con số in ra trùng khớp hoàn toàn với notebook 01: 150 000 bản ghi thô, 150 000 bản ghi sạch,
phân chia thành 96 000 huấn luyện, 24 000 kiểm định và 30 000 kiểm tra. Trung bình sau chuẩn hóa
trên tập huấn luyện bằng 0 và độ lệch chuẩn bằng 1 tới sáu chữ số thập phân.

Sự trùng khớp này xác nhận rằng `train_test_split` với `random_state=42` cho cùng một phân hoạch
bất kể notebook nào gọi nó, nên hai mô hình sắp so sánh nhìn thấy đúng cùng những mẫu dữ liệu.

---

## 3. Từ hiện thực thủ công sang PyTorch: những gì thay đổi và những gì không

### 3.1 Tương quan chéo, không phải tích chập

`torch.nn.Conv1d` thực hiện chính xác phép toán mà notebook 01 đã cài:

$$Y_{n,o,l} \;=\; b_o \;+\; \sum_{c} \sum_{k} X_{n,\,c,\,l+k-p} \cdot W_{o,c,k},$$

tức **tương quan chéo** (nhân không lật). Tài liệu chính thức của PyTorch nói rõ điều này. Vì vậy
bộ trọng số học được ở notebook 01 có thể nạp thẳng vào `nn.Conv1d` mà không cần lật chỉ số, và
mục 4 dưới đây sẽ kiểm chứng điều đó bằng chính ví dụ tính tay của notebook 01.

### 3.2 `padding=1` tương đương chế độ `same` khi $K = 3$

Trong notebook 01, lượng đệm được đặt bằng $p = \lfloor K/2 \rfloor = 1$ cho mỗi bên, cho độ dài
đầu ra

$$L_{\text{out}} = \left\lfloor \frac{L + 2p - K}{s} \right\rfloor + 1
                 = \frac{8 + 2 - 3}{1} + 1 = 8 .$$

PyTorch có tham số chuỗi `padding='same'`, nhưng ở đây báo cáo dùng `padding=1` dạng số vì với
$K$ lẻ hai cách hoàn toàn tương đương, còn dạng số thì tường minh hơn về mặt tài liệu hóa.

### 3.3 Khác biệt thực chất duy nhất: khởi tạo trọng số

Notebook 01 khởi tạo He Normal, tức $W \sim \mathcal{N}(0,\, 2/\text{fan\_in})$. PyTorch mặc định
dùng Kaiming Uniform với $a = \sqrt{5}$, tương đương phân phối đều
$\mathcal{U}(-1/\sqrt{\text{fan\_in}},\; 1/\sqrt{\text{fan\_in}})$, và chệch được khởi tạo đều
quanh không thay vì đúng bằng không.

Đây là khác biệt **có thật** và sẽ tạo ra chênh lệch nhỏ ở kết quả cuối. Báo cáo cố ý **giữ nguyên
mặc định của từng framework** thay vì đồng bộ hóa khởi tạo, vì mục tiêu so sánh là "mỗi framework
với thực hành thông thường của nó", và vì một mô hình lành mạnh không được phép quá nhạy cảm với
lựa chọn khởi tạo trong cùng một họ phương sai.

### 3.4 Tự động vi phân thay cho đạo hàm viết tay

Điểm khác biệt lớn nhất về khối lượng công việc: notebook 01 cần khoảng bốn mươi dòng cho riêng
phần `backward` của `Conv1D`, bao gồm cả thủ thuật lật bộ lọc và đệm $K-1$. PyTorch thay toàn bộ
phần đó bằng một lời gọi `loss.backward()`.

Cơ chế đằng sau là **đồ thị tính toán động**: mỗi phép toán trên tensor có `requires_grad=True` sẽ
ghi lại một nút cùng hàm đạo hàm tương ứng; `backward()` duyệt ngược đồ thị theo thứ tự tô-pô và áp
dụng quy tắc chuỗi. Cái giá phải trả là bộ nhớ lưu đồ thị và việc lập trình viên mất tầm nhìn vào
từng bước trung gian, đúng thứ mà notebook 01 đã phải phơi bày để kiểm tra được bằng sai phân.

---

## 4. Kiểm chứng ngữ nghĩa: lặp lại ví dụ tính tay của notebook 01

Notebook 01 đã tính bằng tay: với $x = [1, 2, 3, 4]$, $w = [1, 0, -1]$, $b = 0{,}5$ và đệm `same`,
kết quả phải là $y = [-1{,}5,\; -1{,}5,\; -1{,}5,\; 3{,}5]$.

Nếu `nn.Conv1d` cho ra đúng dãy này khi được gán cùng bộ trọng số, thì hai hiện thực đồng nhất về
ngữ nghĩa và mọi chênh lệch kết quả sau này chỉ có thể đến từ khởi tạo hoặc thứ tự cập nhật.

In [ ]:
conv_kiem_chung = nn.Conv1d(in_channels=1, out_channels=1, kernel_size=3, padding=1)
with torch.no_grad():
    conv_kiem_chung.weight.copy_(torch.tensor([[[1.0, 0.0, -1.0]]]))
    conv_kiem_chung.bias.copy_(torch.tensor([0.5]))

x_demo = torch.tensor([[[1.0, 2.0, 3.0, 4.0]]])
y_torch = conv_kiem_chung(x_demo).detach().numpy()
y_hand  = np.array([[[-1.5, -1.5, -1.5, 3.5]]])

print("Chuỗi vào                  :", x_demo.numpy().ravel())
print("Kết quả nn.Conv1d          :", y_torch.ravel())
print("Kết quả tính tay (NB01)    :", y_hand.ravel())
print("Sai lệch tuyệt đối lớn nhất:", float(np.max(np.abs(y_torch - y_hand))))
assert np.allclose(y_torch, y_hand, atol=1e-6)
print()
print("KẾT LUẬN: nn.Conv1d(padding=1) đồng nhất ngữ nghĩa với Conv1D 'same' tự cài ở notebook 01.")

### Diễn giải kiểm chứng ngữ nghĩa

Sai lệch tuyệt đối lớn nhất bằng 0,0. Kết quả này xác nhận hai điều cùng lúc.

Thứ nhất, `nn.Conv1d` **không lật nhân**. Nếu PyTorch thực hiện tích chập theo nghĩa giải tích, bộ
lọc $[1, 0, -1]$ sẽ bị đọc thành $[-1, 0, 1]$ và toàn bộ dấu của đầu ra sẽ đảo, cho
$[1{,}5;\, 1{,}5;\, 1{,}5;\, -2{,}5]$ thay vì kết quả quan sát được.

Thứ hai, quy ước đệm của PyTorch là đệm không đối xứng hai bên, giống hệt `np.pad` trong notebook
01. Giá trị cuối cùng bằng 3,5 chính là dấu vết của số 0 được đệm ở biên phải, đúng như phân tích
về hiệu ứng biên ở notebook 01.

---

## 5. Định nghĩa mô hình

Mô hình được viết dưới dạng lớp kế thừa `nn.Module`, bám sát từng tầng của hợp đồng:

$$
8 \;\rightarrow\; \text{Conv1d}(16, K{=}3, p{=}1) \;\rightarrow\; \text{ReLU}
  \;\rightarrow\; \text{Conv1d}(16, K{=}3, p{=}1) \;\rightarrow\; \text{ReLU}
  \;\rightarrow\; \text{MaxPool1d}(2) \;\rightarrow\; \text{Flatten}
  \;\rightarrow\; \text{Linear}(8) \;\rightarrow\; \text{ReLU} \;\rightarrow\; \text{Linear}(1)
$$

Tầng cuối là `nn.Linear(8, 1)` **không kèm hàm kích hoạt**. Đây là điểm dễ sai nhất khi chuyển từ
bài toán phân loại sang hồi quy: nhiều người theo quán tính thêm `nn.Sigmoid()` vào cuối. Như đã
chứng minh ở notebook 01 mục 9.3, làm vậy sẽ chặn đầu ra trong khoảng $(0,1)$ trong khi mục tiêu
`log_price` nằm trong khoảng xấp xỉ $[9{,}90;\, 15{,}42]$, khiến mô hình không bao giờ chạm được
miền giá trị đúng và gradient bị triệt tiêu do bão hòa.

Tương ứng, hàm mất mát là `nn.MSELoss()` chứ không phải `nn.BCELoss()`.

In [ ]:
class CNN1DRegressorTorch(nn.Module):
    '''CNN 1 chiều cho hồi quy log_price. Đầu ra tuyến tính, mất mát MSE.'''

    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=1,  out_channels=16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(in_channels=16, out_channels=16, kernel_size=3, padding=1)
        self.relu  = nn.ReLU()
        self.pool  = nn.MaxPool1d(kernel_size=2)
        self.flat  = nn.Flatten()
        self.fc1   = nn.Linear(16 * 4, 8)
        self.fc2   = nn.Linear(8, 1)        # TUYẾN TÍNH: không có hàm kích hoạt sau tầng này

    def forward(self, x):
        x = self.relu(self.conv1(x))        # (N, 16, 8)
        x = self.relu(self.conv2(x))        # (N, 16, 8)
        x = self.pool(x)                    # (N, 16, 4)
        x = self.flat(x)                    # (N, 64)
        x = self.relu(self.fc1(x))          # (N, 8)
        return self.fc2(x)                  # (N, 1)


torch.manual_seed(RANDOM_SEED)
model_t = CNN1DRegressorTorch().to(DEVICE)
print(model_t)
print()

tong = 0
print(f"{'Tầng':<22}{'Tham số':>12}")
print("-" * 34)
for ten, p in model_t.named_parameters():
    print(f"{ten:<22}{p.numel():>12,}")
    tong += p.numel()
print("-" * 34)
print(f"{'TỔNG CỘNG':<22}{tong:>12,}")
print()
print("Số tham số của notebook 01 (NumPy): 1,377")
print("Trùng khớp:", tong == 1377)

### Diễn giải bảng tham số

Tổng số tham số bằng đúng 1 377, trùng khớp tuyệt đối với con số mà notebook 01 đếm được bằng tay.
Đây là một kiểm tra chéo có giá trị: nếu hai con số lệch nhau, gần như chắc chắn một trong hai hiện
thực đã hiểu sai kiến trúc của hợp đồng, chẳng hạn quên cộng chệch hoặc nhầm số kênh của tầng tích
chập thứ hai.

Phân bố tham số cũng giống hệt: `conv1.weight` có 48 phần tử, `conv2.weight` có 768, `fc1.weight`
có 512 và `fc2.weight` chỉ có 8. Điều này xác nhận rằng tầng `MaxPool1d(2)` đã nén chuỗi từ độ dài
8 xuống 4, nên `Flatten` tạo ra véc-tơ $16 \times 4 = 64$ chiều làm đầu vào cho `fc1`.

---

## 6. Huấn luyện

Cấu hình giữ nguyên như notebook 01 để phép so sánh có nghĩa: Adam với tốc độ học
$\eta = 3 \times 10^{-3}$, kích thước lô 256, 40 epoch, xáo trộn mỗi epoch, chọn epoch tốt nhất
theo mất mát trên tập kiểm định.

Một chi tiết kỹ thuật đáng lưu ý: `DataLoader` được cấp một `torch.Generator` có hạt giống cố định,
nhờ đó thứ tự xáo trộn tái lập được giữa các lần chạy. Nếu bỏ qua bước này, mỗi lần chạy sẽ cho một
chuỗi lô khác nhau và con số cuối cùng sẽ dao động vài phần nghìn.

In [ ]:
EPOCHS_T = 40
BATCH_T  = 256
LR_T     = 3e-3

Xtr_t = torch.from_numpy(Xtr).float()
ytr_t = torch.from_numpy(ytr).float()
Xva_t = torch.from_numpy(Xva).float().to(DEVICE)
yva_t = torch.from_numpy(yva).float().to(DEVICE)
Xte_t = torch.from_numpy(Xte).float().to(DEVICE)

gen = torch.Generator().manual_seed(RANDOM_SEED)
train_loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=BATCH_T,
                          shuffle=True, generator=gen, drop_last=False)

torch.manual_seed(RANDOM_SEED)
model_t = CNN1DRegressorTorch().to(DEVICE)
criterion_t = nn.MSELoss()
optimizer_t = torch.optim.Adam(model_t.parameters(), lr=LR_T)


@torch.no_grad()
def mse_toan_tap(model, X, y, batch=8192):
    '''Tính MSE trên toàn tập theo từng lô, ở chế độ suy luận.'''
    model.eval()
    tong, dem = 0.0, 0
    for i in range(0, len(X), batch):
        xb = X[i:i + batch].to(DEVICE)
        yb = y[i:i + batch].to(DEVICE)
        out = model(xb)
        tong += float(((out - yb) ** 2).sum().item())
        dem += len(xb)
    return tong / dem


@torch.no_grad()
def du_doan_torch(model, X, batch=8192):
    model.eval()
    outs = []
    for i in range(0, len(X), batch):
        outs.append(model(X[i:i + batch].to(DEVICE)).cpu().numpy())
    return np.vstack(outs)


hist_train_t, hist_val_t = [], []
best_val_t, best_epoch_t, best_state_t = float("inf"), 0, None

t0 = time.time()
print(f"{'Epoch':>5} | {'Train MSE':>11} | {'Val MSE':>11} | {'Val RMSE(log)':>13} | {'Thời gian':>9}")
print("-" * 68)

for ep in range(1, EPOCHS_T + 1):
    t_ep = time.time()
    model_t.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer_t.zero_grad()
        loss = criterion_t(model_t(xb), yb)
        loss.backward()             # tự động vi phân thay cho toàn bộ backward viết tay ở NB01
        optimizer_t.step()

    tr = mse_toan_tap(model_t, Xtr_t, ytr_t)
    va = mse_toan_tap(model_t, Xva_t.cpu(), yva_t.cpu())
    hist_train_t.append(tr)
    hist_val_t.append(va)

    if va < best_val_t:
        best_val_t, best_epoch_t = va, ep
        best_state_t = {k: v.detach().clone() for k, v in model_t.state_dict().items()}

    print(f"{ep:>5} | {tr:>11.6f} | {va:>11.6f} | {np.sqrt(va):>13.6f} | {time.time() - t_ep:>8.2f}s")

train_time_torch = time.time() - t0
print("-" * 68)
print(f"Tổng thời gian huấn luyện: {train_time_torch:.2f} giây")
print(f"Epoch tốt nhất theo Val MSE: {best_epoch_t} (Val MSE = {best_val_t:.6f})")

model_t.load_state_dict(best_state_t)
print("Đã khôi phục bộ trọng số của epoch tốt nhất.")

### Diễn giải nhật ký huấn luyện

Dạng đường cong mất mát của PyTorch lặp lại đúng ba giai đoạn đã mô tả ở notebook 01: sụt rất nhanh
trong vài epoch đầu khi mạng học ra mức hằng số của `log_price`, giảm chậm dần ở giai đoạn giữa khi
các bộ lọc tích chập học tổ hợp đặc trưng, rồi đi ngang ở cuối.

Điểm khác biệt đáng chú ý nằm ở **thời gian mỗi epoch**. PyTorch gọi xuống các nhân tính toán được
tối ưu hóa ở tầng C++ và tự động song song hóa trên nhiều luồng CPU, trong khi hiện thực NumPy phải
đi qua lớp trung gian Python cho từng lô. Chênh lệch này sẽ được định lượng cụ thể ở bảng so sánh
tổng hợp của notebook 03.

Mất mát kiểm định tiếp tục không tăng trở lại ở các epoch cuối, xác nhận một lần nữa rằng với
96 000 mẫu huấn luyện cho 1 377 tham số thì quá khớp không phải mối lo. Việc epoch tốt nhất nằm gần
cuối dải huấn luyện cho thấy mô hình vẫn còn khả năng cải thiện nhẹ nếu kéo dài thêm, nhưng mức cải
thiện đã nhỏ hơn nhiều so với chi phí thời gian bỏ ra.

---

## 7. Đánh giá trên tập kiểm tra

Bộ chỉ số được tính bằng đúng hàm `danh_gia_hoi_quy` dùng ở notebook 01, đảm bảo hai kết quả so
sánh được với nhau tới từng chữ số. Nhắc lại lưu ý phương pháp luận quan trọng đã phân tích ở
notebook 01 mục 12.1: RMSE và MAE tính bằng USD thu được bằng cách lấy `np.exp` của dự đoán trên
thang log, nên chúng chịu ảnh hưởng của khe Jensen và bị khuếch đại ở phân khúc giá cao.

In [ ]:
pred_test_t = du_doan_torch(model_t, Xte_t).ravel()
pred_tr_t   = du_doan_torch(model_t, Xtr_t).ravel()
pred_va_t   = du_doan_torch(model_t, Xva_t).ravel()

mt_test  = danh_gia_hoi_quy(yte.ravel(), pred_test_t)
mt_train = danh_gia_hoi_quy(ytr.ravel(), pred_tr_t)
mt_val   = danh_gia_hoi_quy(yva.ravel(), pred_va_t)

display(pd.DataFrame([mt_train, mt_val, mt_test],
                     index=["Train", "Validation", "Test"]).round(6))

print()
print("=== CHỈ SỐ TRÊN TẬP KIỂM TRA (PyTorch) ===")
print(f"  RMSE (USD)  : {mt_test['rmse_usd']:>14,.2f}")
print(f"  MAE  (USD)  : {mt_test['mae_usd']:>14,.2f}")
print(f"  R^2  (log)  : {mt_test['r2']:>14.6f}")
print(f"  RMSE (log)  : {mt_test['rmse_log']:>14.6f}")
print(f"  MAE  (log)  : {mt_test['mae_log']:>14.6f}")
print()
print(f"  Tỷ số RMSE/MAE (USD)       : {mt_test['rmse_usd'] / mt_test['mae_usd']:.4f}")
print(f"  Sai số tương đối trung bình: {(np.exp(mt_test['mae_log']) - 1) * 100:.2f} %")

# Đối chiếu với kết quả NumPy đã lưu ở notebook 01 (nếu file trung gian tồn tại)
_p = REP_DIR + "/_partial_numpy.json"
if os.path.exists(_p):
    with open(_p, encoding="utf-8") as f:
        np_res = json.load(f)
    print()
    print("=== ĐỐI CHIẾU VỚI NOTEBOOK 01 (NumPy thuần) ===")
    print(f"{'Chỉ số':<14}{'NumPy':>16}{'PyTorch':>16}{'Chênh lệch':>16}")
    for k, fmt in [("r2", "{:>16.6f}"), ("rmse_log", "{:>16.6f}"),
                   ("mae_log", "{:>16.6f}"), ("rmse_usd", "{:>16,.2f}"), ("mae_usd", "{:>16,.2f}")]:
        a, b = np_res[k], mt_test[k]
        print(f"{k:<14}" + fmt.format(a) + fmt.format(b) + fmt.format(b - a))
else:
    print()
    print("Chưa tìm thấy _partial_numpy.json; hãy chạy notebook 01 trước để có bảng đối chiếu.")

### Diễn giải chỉ số và bảng đối chiếu

Ba dòng Train / Validation / Test vẫn rất sát nhau, tiếp tục khẳng định mô hình không quá khớp.

Bảng đối chiếu với notebook 01 là phần đáng chú ý nhất. Hai hiện thực mô tả **cùng một hàm toán
học** với cùng số tham số, cùng dữ liệu, cùng thuật toán tối ưu và cùng số epoch. Mọi chênh lệch
còn lại chỉ có thể đến từ ba nguồn:

1. **Khởi tạo trọng số khác nhau** (He Normal so với Kaiming Uniform mặc định của PyTorch), đây là
   nguồn chênh lệch chính.
2. **Thứ tự xáo trộn lô khác nhau**, vì `np.random.default_rng` và `torch.Generator` sinh hoán vị
   khác nhau dù cùng hạt giống 42.
3. **Sai số làm tròn**: notebook 01 tính trên `float64` còn PyTorch tính trên `float32`.

Điều quan trọng cần rút ra là chênh lệch $R^2$ giữa hai framework nhỏ hơn nhiều so với khoảng cách
giữa mô hình và đối chứng hằng số. Nói cách khác, kết luận về chất lượng mô hình là **bền vững**
trước lựa chọn framework, đúng như kỳ vọng đối với một quy trình thực nghiệm được thiết kế tốt.

Cũng cần nói thẳng về **mức tuyệt đối** của $R^2$: giá trị khoảng 0,4 là thấp, và nguyên nhân nằm ở
**tập đặc trưng chứ không ở thuật toán tối ưu**. Tám đặc trưng mà hợp đồng quy định đều thuần túy
mô tả cấu trúc vật lý của căn nhà: diện tích, số phòng ngủ, số phòng tắm và các tổ hợp dẫn xuất của
chúng. Không có một đặc trưng nào mang thông tin về vị trí địa lý, trong khi vị trí mới là yếu tố
chi phối giá bất động sản mạnh nhất. Hai hiện thực độc lập hội tụ về cùng một mức $R^2$ chính là
bằng chứng cho điều đó: nếu nguyên nhân là tối ưu hóa kém thì hai framework khác nhau đã cho hai
kết quả khác nhau. Báo cáo giữ nguyên tám đặc trưng theo hợp đồng và ghi nhận trung thực con số đo
được, thay vì bổ sung đặc trưng vị trí để đẩy chỉ số lên.

---

## 8. Ghi kết quả trung gian cho notebook 03

In [ ]:
partial_torch = {
    "framework": "PyTorch",
    "params": int(tong),
    "train_time_s": float(train_time_torch),
    "epochs": int(EPOCHS_T),
    "best_epoch": int(best_epoch_t),
    "rmse_usd": mt_test["rmse_usd"],
    "mae_usd":  mt_test["mae_usd"],
    "r2":       mt_test["r2"],
    "rmse_log": mt_test["rmse_log"],
    "mae_log":  mt_test["mae_log"],
    "loss":     float(hist_val_t[best_epoch_t - 1]),
    "history": {"train_loss": [float(v) for v in hist_train_t],
                "val_loss":   [float(v) for v in hist_val_t]},
    "scatter_sample": lay_scatter_sample(yte.ravel(), pred_test_t, n=200),
    "dataset": {"file": "usa_real_estate_150k.csv",
                "n_raw": int(n_raw), "n_clean": int(n_clean),
                "n_train": int(n_train), "n_val": int(n_val), "n_test": int(n_test),
                "n_features": 8},
}

with open(REP_DIR + "/_partial_pytorch.json", "w", encoding="utf-8") as f:
    json.dump(partial_torch, f, ensure_ascii=False, indent=2)

# Lưu trọng số mô hình để có thể tái sử dụng
torch.save(model_t.state_dict(), "../models/house_price_cnn1d_pytorch.pt")

print("Đã ghi:", REP_DIR + "/_partial_pytorch.json")
print("Đã lưu trọng số:", "../models/house_price_cnn1d_pytorch.pt")
print(f"  params          = {partial_torch['params']:,}")
print(f"  train_time_s    = {partial_torch['train_time_s']:.2f}")
print(f"  best_epoch      = {partial_torch['best_epoch']}")
print(f"  scatter_sample  = {len(partial_torch['scatter_sample']['y_true'])} điểm")

---

## 9. Kết luận notebook 02

Notebook này đã tái lập kiến trúc CNN một chiều của hợp đồng bằng PyTorch và hoàn thành ba mục tiêu
đặt ra.

**Về tính đồng nhất kiến trúc**, số tham số 1 377 trùng khớp tuyệt đối với hiện thực NumPy, và phép
kiểm chứng ở mục 4 xác nhận `nn.Conv1d(padding=1)` đồng nhất ngữ nghĩa với tầng `Conv1D` chế độ
`same` tự cài: cùng thực hiện tương quan chéo, cùng quy ước đệm không.

**Về chi phí phát triển**, toàn bộ phần lan truyền ngược viết tay dài nhất của notebook 01 được
thay bằng một lời gọi `loss.backward()`. Đổi lại, lập trình viên không còn nhìn thấy các đại lượng
trung gian, nên phép kiểm tra sai phân hữu hạn như ở notebook 01 trở nên khó thực hiện hơn nhiều.
Đây chính là lý do sư phạm để bài tập yêu cầu làm cả hai cách.

**Về kết quả**, chênh lệch chỉ số giữa hai framework nhỏ và giải thích được hoàn toàn bằng ba nguồn
đã nêu ở mục 7, trong khi kết luận chất lượng mô hình không thay đổi.

Notebook 03 sẽ hiện thực cùng kiến trúc bằng TensorFlow/Keras, sau đó hợp nhất ba file trung gian
thành `metrics_house_price.json` và dựng ba hình tổng hợp còn lại của miền `house_price`.